## 1. Cai Dat Dependencies

In [ ]:
import torch;
print(torch.__version__);
print(torch.version.cuda);
print(torch.cuda.is_available());
print(torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "No GPU available"

In [ ]:
# Cài đặt các thư viện cần thiết
# %pip install torch torchvision transformers accelerate pillow einops timm gradio sentencepiece safetensors requests matplotlib

In [ ]:
# Import các thư viện
import torch
import torch.nn as nn
from PIL import Image
from transformers import AutoModel, AutoModelForCausalLM, AutoImageProcessor, AutoTokenizer
from dataclasses import dataclass
from typing import Optional, List
import requests
from io import BytesIO

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

---
## 1.5. Demo Dataset Structure

Xem cau truc du lieu truoc khi bat dau.

In [ ]:
# ============================================
# DEMO DATASET STRUCTURE
# ============================================

import json
import os
from PIL import Image
import matplotlib.pyplot as plt

# Duong dan dataset
DATA_DIR = "datasets"
IMAGE_DIR = os.path.join(DATA_DIR, "image")

# Load 1 sample tu train.json
with open(os.path.join(DATA_DIR, "train.json"), 'r', encoding='utf-8') as f:
    train_data = json.load(f)

print(f"Total training samples: {len(train_data)}")
print("\n" + "=" * 80)
print("SAMPLE DATA STRUCTURE:")
print("=" * 80)

# Hien thi 1 sample
sample = train_data[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"\nComment: {sample.get('comment', 'N/A')}")
print(f"\nImage files: {sample.get('list_img', [])}")
print(f"\nLabels (text_img_label): {sample.get('text_img_label', [])}")

# Load va hien thi anh dau tien (neu co)
if sample.get('list_img') and len(sample['list_img']) > 0:
    img_path = os.path.join(IMAGE_DIR, sample['list_img'][0])
    if os.path.exists(img_path):
        print(f"\nImage path: {img_path}")
        
        # Hien thi anh
        img = Image.open(img_path)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Sample Image\nLabels: {sample.get('text_img_label', [])}")
        plt.tight_layout()
        plt.show()
        
        print(f"\nImage size: {img.size}")
        print(f"Image mode: {img.mode}")
    else:
        print(f"\n[WARNING] Image not found: {img_path}")

print("\n" + "=" * 80)
print("LABEL DISTRIBUTION:")
print("=" * 80)

# Dem phan bo labels
from collections import Counter
all_labels = []
for item in train_data:
    labels = item.get('text_img_label', [])
    all_labels.extend(labels)

label_counts = Counter(all_labels)
print(f"\nTotal labels: {len(all_labels)}")
print(f"Unique labels: {len(label_counts)}")
print(f"\nTop 10 most common labels:")
for label, count in label_counts.most_common(10):
    print(f"  {label:<40} {count:>5} ({count/len(all_labels)*100:.1f}%)")

---
## 2. Preprocessor và Swin Transformer V2 (Vision Encoder)

**Preprocessor**: Chuyen doi anh sang tensor, resize ve 256x256
**Swin Transformer V2**: Su dung hierarchical vision transformer voi shifted windows de xu ly anh hieu qua hon.

| Thong so | Gia tri |
|----------|--------|
| Model | `microsoft/swinv2-base-patch4-window8-256` |
| Input | 256x256 |
| Patch Size | 4x4 |
| Window Size | 8x8 |
| Hidden Dim | 1024 |
| Num Tokens | 64 (8x8 spatial features) |

In [ ]:
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

# ============================================
# PREPROCESSOR - Chuyen doi anh sang tensor
# ============================================

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size=256):
    """
    Preprocessor: Image -> Tensor
    
    Input: PIL Image (any size)
    Output: Tensor [3, input_size, input_size] normalized
    """
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

# ============================================
# SWIN TRANSFORMER V2 - Vision Encoder
# ============================================

class VisionEncoder(nn.Module):
    """
    Swin Transformer V2: Image -> Hierarchical features -> Img tokens
    
    Input: Tensor [B, 3, 256, 256]
    Output: Img tokens [B, 64, 1024]
    """
    
    def __init__(
        self, 
        model_name: str = "microsoft/swinv2-base-patch4-window8-256",
        device: str = "cuda",
        torch_dtype: torch.dtype = torch.float16
    ):
        super().__init__()
        self.device = device
        self.torch_dtype = torch_dtype
        
        print(f"Loading Swin Transformer V2: {model_name}")
        self.model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch_dtype
        ).to(device)
        
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        
        # Freeze weights
        for param in self.model.parameters():
            param.requires_grad = False
        self.model.eval()
        
        # Swin Transformer V2 Base: hidden_size = 1024
        self.hidden_size = self.model.config.hidden_size  # 1024
        # Output spatial size for 256x256 input: 8x8 = 64 tokens
        self.num_patches = 64  # 8x8 spatial features at final stage
        
        print(f"Loaded: hidden_size={self.hidden_size}, num_patches={self.num_patches}")
    
    @torch.no_grad()
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Extract visual features using Swin Transformer V2.
        
        Input: [B, 3, 256, 256]
        Output: [B, 64, 1024] - Img tokens
        """
        pixel_values = pixel_values.to(self.device, dtype=self.torch_dtype)

        outputs = self.model(pixel_values=pixel_values)
        return outputs.last_hidden_state  # [B, 64, 1024]
        # Swin outputs: last_hidden_state shape [B, 64, 1024] for base model

---
## 3. Projection Layer (MLP)

Chuyen doi Img tokens tu Swin Transformer V2 (1024 dim) sang Lang tokens cho LLM (4096 dim).

```
Img tokens [B, 64, 1024] -> Linear(1024, 4096) -> GELU -> Linear(4096, 4096) -> Lang tokens [B, 64, 4096]
```

In [ ]:
class MLPProjector(nn.Module):
    """
    Projection: Img tokens -> Lang tokens
    
    Input: [B, 64, 1024] tu Swin Transformer V2
    Output: [B, 64, 4096] cho LLM
    """
    
    def __init__(
        self,
        vision_dim: int = 1024,   # Swin Transformer V2 hidden dim
        llm_dim: int = 4096,      # LLM hidden dim (InternLM3-8B = 4096)
    ):
        super().__init__()
        
        self.projector = nn.Sequential(
            nn.LayerNorm(vision_dim),
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )
        
        # Initialize weights
        for module in self.projector:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
    
    def forward(self, img_tokens: torch.Tensor) -> torch.Tensor:
        """
        Project Img tokens to Lang tokens.
        
        Input: [B, 64, 1024]
        Output: [B, 64, 4096]
        """
        # Cast input to match projector weight dtype (float16 -> float32)
        img_tokens = img_tokens.to(dtype=self.projector[0].weight.dtype)
        return self.projector(img_tokens)

In [ ]:
# ============================================
# KHOI TAO VISION ENCODER VA PROJECTOR
# ============================================

# Tao Vision Encoder - Swin Transformer V2
vision_encoder = VisionEncoder(
    model_name="microsoft/swinv2-base-patch4-window8-256",
    device=device,
    torch_dtype=torch.float16
)

# Tao Projection Layer - chuyen tu Swin features sang LLM space
# InternLM3-8B hidden_size = 4096
projector = MLPProjector(
    vision_dim=1024,   # Swin V2 Base hidden dim
    llm_dim=4096,      # InternLM3-8B hidden dim
).to(device)

print(f"  Vision hidden dim: {vision_encoder.hidden_size}")
print(f"  Vision num patches: {vision_encoder.num_patches}")
print(f"  Projector trainable params: {sum(p.numel() for p in projector.parameters()):,}")

---
## 4. LLM - InternLM3-8B-Instruct

InternLM3 la pure LLM (khong co ViT nhu InternVL3), tiet kiem ~6GB VRAM.

LLM nhan:
- Lang tokens tu Projection (Image path)
- Lang tokens tu Embeddings (Text path)

| Model | Parameters | VRAM |
|-------|------------|------|
| InternLM3-8B-Instruct | 8B | ~16GB |

In [ ]:
# ============================================
# SET HUGGING FACE CACHE TO F: DRIVE
# (Tranh download vao C:\Users\.cache\huggingface)
# ============================================
import os
os.environ["HF_HOME"] = r"F:\HuggingFaceCache"
print(f"HF_HOME set to: {os.environ['HF_HOME']}")

In [ ]:
# ============================================
# LOAD LLM - InternLM3-8B-Instruct
# ============================================
# InternLM3 la pure LLM (khong co ViT nhu InternVL3)
# -> Khong can FIX patch cho InternViT
# -> Khong load ViT thua, tiet kiem ~6GB VRAM
# -> Truy cap truc tiep model (khong can .language_model)

import sys
import os
from pathlib import Path
from typing import Optional

# ============================================
# PATCH: InternLM3 remote code imports LossKwargs
# from transformers.utils, but it doesn't exist
# in any stable transformers release.
# We inject a minimal stub so the import succeeds.
# ============================================
import transformers.utils as _tu
if not hasattr(_tu, "LossKwargs"):
    from typing import TypedDict
    class LossKwargs(TypedDict, total=False):
        num_items_in_batch: Optional[int]
    _tu.LossKwargs = LossKwargs
    print("[PATCH] Injected LossKwargs into transformers.utils")

MODEL_NAME = "internlm/internlm3-8b-instruct"

print(f"Loading {MODEL_NAME}...")
print("Lan dau se download ~16GB, vui long cho...")

# Determine dtype
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

# Load tokenizer (Text -> word-pieces -> Embeddings -> Lang tokens)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# ============================================
# Load model (Pure LLM - khong co ViT)
# Su dung AutoModelForCausalLM thay vi AutoModel
# ============================================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    trust_remote_code=True,
).to(device).eval()

print(f"Model loaded!")
print(f"  Model type: {type(model).__name__}")
print(f"  Hidden size: {model.config.hidden_size}")
print(f"  Num layers: {model.config.num_hidden_layers}")
print(f"  Dtype: {torch_dtype}")

---
## 5. Dataset va DataLoader

In [ ]:
import json
import os
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ============================================
# LABEL DEFINITIONS - Aspect Categories + Sentiments (Dual-Head)
# ============================================

# 6 Aspect Categories
ASPECT_LABELS = ["Facilities", "Public_area", "Location", "Food", "Room", "Service"]
ASPECT2ID = {label: idx for idx, label in enumerate(ASPECT_LABELS)}
ID2ASPECT = {idx: label for idx, label in enumerate(ASPECT_LABELS)}
NUM_ASPECTS = len(ASPECT_LABELS)

# 3 Sentiments (per aspect)
SENTIMENT_LABELS = ["Positive", "Negative", "Neutral"]
SENTIMENT2ID = {label: idx for idx, label in enumerate(SENTIMENT_LABELS)}
ID2SENTIMENT = {idx: label for idx, label in enumerate(SENTIMENT_LABELS)}
NUM_SENTIMENTS = len(SENTIMENT_LABELS)

# Combined Aspect#Sentiment labels (18 classes) - for evaluation
COMBINED_LABELS = [f"{a}#{s}" for a in ASPECT_LABELS for s in SENTIMENT_LABELS]
COMBINED2ID = {label: idx for idx, label in enumerate(COMBINED_LABELS)}
ID2COMBINED = {idx: label for idx, label in enumerate(COMBINED_LABELS)}
NUM_COMBINED = len(COMBINED_LABELS)

# Backward compatibility
LABEL2ID = SENTIMENT2ID
ID2LABEL = ID2SENTIMENT
NUM_LABELS = NUM_SENTIMENTS

print(f"Aspect Categories ({NUM_ASPECTS}): {ASPECT_LABELS}")
print(f"Sentiments ({NUM_SENTIMENTS}): {SENTIMENT_LABELS}")
print(f"Combined Labels ({NUM_COMBINED}): {COMBINED_LABELS[:6]}...")


In [ ]:
# ============================================
# DATA LOADER - Doc train/dev/test.json
# ============================================

def load_dataset_json(json_path: str) -> list:
    """
    Doc file JSON dataset.
    
    Args:
        json_path: Duong dan toi file JSON
        
    Returns:
        List cac samples
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"Loaded {len(data)} samples from {json_path}")
    return data


def load_all_splits(data_dir: str = "datasets"):
    """
    Load tat ca cac split: train, dev, test.
    
    Args:
        data_dir: Thu muc chua data
        
    Returns:
        Dict chua train, dev, test data
    """
    splits = {}
    for split in ["train", "dev", "test"]:
        json_path = os.path.join(data_dir, f"{split}.json")
        if os.path.exists(json_path):
            splits[split] = load_dataset_json(json_path)
        else:
            print(f"[WARNING] {json_path} not found!")
            splits[split] = []
    return splits


# Load data
DATA_DIR = "datasets"
IMAGE_DIR = os.path.join(DATA_DIR, "image")

dataset_splits = load_all_splits(DATA_DIR)

print(f"\nDataset Statistics:")
print(f"  Train: {len(dataset_splits['train'])} samples")
print(f"  Dev:   {len(dataset_splits['dev'])} samples")
print(f"  Test:  {len(dataset_splits['test'])} samples")

In [ ]:
# ============================================
# PYTORCH DATASET CLASS - Dual-Head (Aspect + Sentiment)
# ============================================

class SentimentDataset(Dataset):
    """
    PyTorch Dataset cho Aspect-Category Sentiment Analysis (Dual-Head).
    
    Moi sample gom:
    - comment: Text binh luan
    - list_img: Danh sach ten file anh
    - aspect_labels: Multi-hot [6] - aspect nao xuat hien
    - sentiment_labels: [6, 3] - per-aspect sentiment (multi-hot cho moi aspect)
    
    VD: ["Room#Positive", "Service#Negative"]
    -> aspect_labels:    [0, 0, 0, 0, 1, 1]  (Room=1, Service=1)
    -> sentiment_labels: [[0,0,0], [0,0,0], [0,0,0], [0,0,0], [1,0,0], [0,1,0]]
                          Facilities  Public   Location  Food    Room:Pos  Service:Neg
    """
    
    def __init__(
        self,
        data: list,
        image_dir: str,
        aspect2id: dict,
        sentiment2id: dict,
        transform=None,
    ):
        self.data = data
        self.image_dir = image_dir
        self.aspect2id = aspect2id
        self.sentiment2id = sentiment2id
        self.num_aspects = len(aspect2id)
        self.num_sentiments = len(sentiment2id)
        self.transform = transform if transform else build_transform(256)
        
        self.samples = self._prepare_samples()
        print(f"Prepared {len(self.samples)} valid samples")
    
    def _parse_label(self, label: str):
        """
        Parse label goc thanh (aspect, sentiment).
        VD: 'Room#Positive' -> ('Room', 'Positive')
        Returns None neu label khong hop le.
        """
        if "#" not in label:
            return None
        parts = label.split("#")
        if len(parts) != 2:
            return None
        aspect, sentiment = parts[0], parts[1]
        if aspect in self.aspect2id and sentiment in self.sentiment2id:
            return (aspect, sentiment)
        return None
    
    def _prepare_samples(self) -> list:
        """Loc va chuan bi samples hop le."""
        valid_samples = []
        
        for item in self.data:
            if not item.get("list_img") or len(item["list_img"]) == 0:
                continue
            
            raw_labels = item.get("text_img_label", [])
            if not raw_labels:
                continue
            
            # Parse tung label thanh (aspect, sentiment)
            parsed_labels = []
            for label in raw_labels:
                parsed = self._parse_label(label)
                if parsed is not None:
                    parsed_labels.append(parsed)
            
            if not parsed_labels:
                continue
            
            img_name = item["list_img"][0]
            img_path = os.path.join(self.image_dir, img_name)
            if not os.path.exists(img_path):
                continue
            
            valid_samples.append({
                "comment": item.get("comment", ""),
                "image_path": img_path,
                "parsed_labels": parsed_labels,  # List[(aspect, sentiment)]
                "raw_labels": raw_labels,         # Giu lai de debug
            })
        
        return valid_samples
    
    def __len__(self):
        return len(self.samples)
    
    def _labels_to_tensors(self, parsed_labels: list):
        """
        Convert list of (aspect, sentiment) thanh 2 tensors:
        - aspect_labels: multi-hot [num_aspects] = aspect nao xuat hien
        - sentiment_labels: [num_aspects, num_sentiments] = per-aspect sentiment
        """
        aspect_tensor = torch.zeros(self.num_aspects)
        sentiment_tensor = torch.zeros(self.num_aspects, self.num_sentiments)
        
        for aspect, sentiment in parsed_labels:
            a_id = self.aspect2id[aspect]
            s_id = self.sentiment2id[sentiment]
            aspect_tensor[a_id] = 1.0
            sentiment_tensor[a_id][s_id] = 1.0
        
        return aspect_tensor, sentiment_tensor
    
    def __getitem__(self, idx: int) -> dict:
        sample = self.samples[idx]
        
        # Load image
        image = Image.open(sample["image_path"]).convert("RGB")
        
        # Preprocessor: Image -> Tensor
        pixel_values = self.transform(image)
        
        # Convert labels to tensors (dual-head)
        aspect_labels, sentiment_labels = self._labels_to_tensors(sample["parsed_labels"])
        
        return {
            "pixel_values": pixel_values,
            "aspect_labels": aspect_labels,          # [6]
            "sentiment_labels": sentiment_labels,    # [6, 3]
            "comment": sample["comment"],
            "image_path": sample["image_path"],
        }


def collate_fn(batch):
    """
    Custom collate function cho DataLoader (Dual-Head).
    
    Text path: Text -> Tokenizer -> word-pieces (input_ids)
    """
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    aspect_labels = torch.stack([item["aspect_labels"] for item in batch])        # [B, 6]
    sentiment_labels = torch.stack([item["sentiment_labels"] for item in batch])  # [B, 6, 3]
    comments = [item["comment"] for item in batch]
    image_paths = [item["image_path"] for item in batch]
    
    # Tokenizer: Text -> word-pieces (input_ids, attention_mask)
    text_inputs = tokenizer(
        comments,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    
    return {
        "pixel_values": pixel_values,
        "input_ids": text_inputs.input_ids,           # word-pieces
        "attention_mask": text_inputs.attention_mask,
        "aspect_labels": aspect_labels,               # [B, 6]
        "sentiment_labels": sentiment_labels,          # [B, 6, 3]
        "comments": comments,
        "image_paths": image_paths,
    }


In [ ]:
# ============================================
# TAO DATALOADERS (Dual-Head)
# ============================================

# Tao datasets - truyen aspect2id va sentiment2id
train_dataset = SentimentDataset(
    data=dataset_splits["train"],
    image_dir=IMAGE_DIR,
    aspect2id=ASPECT2ID,
    sentiment2id=SENTIMENT2ID,
)

dev_dataset = SentimentDataset(
    data=dataset_splits["dev"],
    image_dir=IMAGE_DIR,
    aspect2id=ASPECT2ID,
    sentiment2id=SENTIMENT2ID,
)

test_dataset = SentimentDataset(
    data=dataset_splits["test"],
    image_dir=IMAGE_DIR,
    aspect2id=ASPECT2ID,
    sentiment2id=SENTIMENT2ID,
)

# Tao dataloaders
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

print(f"\nDataLoader Statistics:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Dev batches:   {len(dev_loader)}")
print(f"  Test batches:  {len(test_loader)}")

# Test 1 batch
sample_batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  pixel_values shape:     {sample_batch['pixel_values'].shape}")
print(f"  aspect_labels shape:    {sample_batch['aspect_labels'].shape}")
print(f"  sentiment_labels shape: {sample_batch['sentiment_labels'].shape}")


---
## 6. Training Loop (Dual-Head: Aspect + Sentiment)

Fine-tune Projection Layer + Dual Classifier Heads, giu nguyen (freeze) Swin Transformer V2 va LLM.

```
Pooled [B, 4096] ──┬──> Aspect Head ──> [B, 6]     (aspect nao xuat hien)
                    │
                    └──> Sentiment Head ──> [B, 6, 3] (sentiment per aspect)
```


In [ ]:
# ============================================
# MULTIMODAL SENTIMENT MODEL - DUAL HEAD
# ============================================
# Architecture:
# Image -> Preprocessor -> Swin V2 -> Projection ──┐
#                                                  ├──> LLM -> Pooling ──┬──> Aspect Head -> [B, 6]
# Text  -> Tokenizer -> Embeddings ────────────────┘                    └──> Sentiment Head -> [B, 6, 3]

class MultimodalSentimentModel(nn.Module):
    """
    Dual-Head Aspect-Category Sentiment Model.
    
    Shared backbone: Swin V2 + Projection + LLM -> Pooled representation [B, 4096]
    
    Head 1 - Aspect Head (multi-label BCE):
      Pooled [B, 4096] -> Aspect Classifier -> [B, num_aspects]
      Detect aspect nao xuat hien (Facilities, Public_area, Location, Food, Room, Service)
    
    Head 2 - Sentiment Head (per-aspect CrossEntropy):
      Pooled [B, 4096] -> Sentiment Classifier -> [B, num_aspects * num_sentiments] -> reshape [B, 6, 3]
      Predict sentiment (Positive/Negative/Neutral) cho moi aspect
      Loss chi tinh tren aspects thuc su xuat hien (masked)
    """
    
    def __init__(
        self,
        vision_encoder: VisionEncoder,
        projector: MLPProjector,
        llm_model,
        tokenizer,
        num_aspects: int = 6,
        num_sentiments: int = 3,
    ):
        super().__init__()
        
        self.vision_encoder = vision_encoder
        self.projector = projector
        self.llm = llm_model
        self.tokenizer = tokenizer
        self.num_aspects = num_aspects
        self.num_sentiments = num_sentiments
        
        # Detect LLM dtype and hidden size
        self.llm_dtype = next(llm_model.parameters()).dtype
        self.llm_hidden_size = llm_model.config.hidden_size  # 4096
        print(f"  LLM dtype: {self.llm_dtype}")
        print(f"  LLM hidden_size: {self.llm_hidden_size}")
        
        # Text Embeddings layer tu LLM (frozen)
        self.text_embeddings = llm_model.get_input_embeddings()
        for param in self.text_embeddings.parameters():
            param.requires_grad = False
        
        # Freeze vision encoder
        for param in self.vision_encoder.parameters():
            param.requires_grad = False
        
        # Freeze LLM
        for param in self.llm.parameters():
            param.requires_grad = False
        
        # Enable gradient checkpointing to save VRAM
        # (gradient van flow qua LLM nhung khong luu tat ca activations)
        self.llm.gradient_checkpointing_enable()
        
        # ============================================
        # ASPECT HEAD (trainable) - Multi-label: aspect nao xuat hien?
        # Pooled [B, 4096] -> [B, num_aspects]
        # ============================================
        self.aspect_classifier = nn.Sequential(
            nn.Linear(self.llm_hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_aspects),
        )
        
        # ============================================
        # SENTIMENT HEAD (trainable) - Per-aspect sentiment
        # Pooled [B, 4096] -> [B, num_aspects * num_sentiments] -> reshape [B, 6, 3]
        # ============================================
        self.sentiment_classifier = nn.Sequential(
            nn.Linear(self.llm_hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_aspects * num_sentiments),
        )
        
        # Initialize both heads
        for head in [self.aspect_classifier, self.sentiment_classifier]:
            for module in head:
                if isinstance(module, nn.Linear):
                    nn.init.xavier_uniform_(module.weight)
                    nn.init.zeros_(module.bias)
        
        print(f"  Aspect Head: {self.llm_hidden_size} -> 256 -> {num_aspects}")
        print(f"  Sentiment Head: {self.llm_hidden_size} -> 256 -> {num_aspects * num_sentiments}")
    
    def _get_combined_embeds(self, pixel_values, input_ids, attention_mask):
        """
        Shared logic: build combined embeddings from image + text.
        
        Returns: combined_embeds, full_attention_mask
        """
        # IMAGE PATH: Image -> Swin V2 -> Projection -> Img Lang tokens
        with torch.no_grad():
            img_tokens = self.vision_encoder(pixel_values)  # [B, 64, 1024]
        
        img_lang_tokens = self.projector(img_tokens)  # [B, 64, 4096]
        
        # TEXT PATH: word-pieces -> Embeddings -> Text Lang tokens
        with torch.no_grad():
            text_lang_tokens = self.text_embeddings(input_ids.to(img_lang_tokens.device))
        
        # Cast to LLM dtype
        img_lang_tokens = img_lang_tokens.to(dtype=self.llm_dtype)
        text_lang_tokens = text_lang_tokens.to(dtype=self.llm_dtype)
        
        # Concatenate
        combined_embeds = torch.cat([img_lang_tokens, text_lang_tokens], dim=1)
        
        # Attention mask
        batch_size = img_lang_tokens.size(0)
        img_mask = torch.ones(batch_size, img_lang_tokens.size(1), device=img_lang_tokens.device)
        if attention_mask is not None:
            full_attention_mask = torch.cat([img_mask, attention_mask.to(img_lang_tokens.device)], dim=1)
        else:
            full_attention_mask = torch.ones(batch_size, combined_embeds.size(1), device=img_lang_tokens.device)
        
        return combined_embeds, full_attention_mask
    
    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor = None,
    ) -> dict:
        """
        Dual-head classification forward pass.
        
        Returns: dict with:
          - aspect_logits: [B, num_aspects]
          - sentiment_logits: [B, num_aspects, num_sentiments]
        """
        combined_embeds, full_attention_mask = self._get_combined_embeds(
            pixel_values, input_ids, attention_mask
        )
        
        # LLM forward -> get hidden states
        with torch.no_grad():
            outputs = self.llm(
                inputs_embeds=combined_embeds,
                attention_mask=full_attention_mask,
                output_hidden_states=True,
            )
            hidden_states = outputs.hidden_states[-1]  # [B, seq_len, 4096]
        
        # Mean pooling over sequence (weighted by attention mask)
        mask = full_attention_mask.unsqueeze(-1).to(hidden_states.dtype)  # [B, seq_len, 1]
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)     # [B, 4096]
        pooled_f32 = pooled.float()  # float32 for stable training
        
        # ASPECT HEAD: [B, 4096] -> [B, num_aspects]
        aspect_logits = self.aspect_classifier(pooled_f32)
        sentiment_logits = sentiment_logits.view(-1, self.num_aspects, self.num_sentiments)
        # SENTIMENT HEAD: [B, 4096] -> [B, num_aspects * num_sentiments] -> [B, 6, 3]
        sentiment_logits = self.sentiment_classifier(pooled_f32)
        sentiment_logits = sentiment_logits.view(-1, self.num_aspects, self.num_sentiments)
        
        return {
            "aspect_logits": aspect_logits,        # [B, 6]
            "sentiment_logits": sentiment_logits,  # [B, 6, 3]
        }
    
    def generate_text(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor = None,
        max_new_tokens: int = 128,
    ) -> list:
        """
        Text generation for inference.
        """
        combined_embeds, full_attention_mask = self._get_combined_embeds(
            pixel_values, input_ids, attention_mask
        )
        
        with torch.no_grad():
            output_ids = self.llm.generate(
                inputs_embeds=combined_embeds,
                attention_mask=full_attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        
        return self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    
    def get_trainable_params(self):
        """Tra ve so luong trainable parameters."""
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable


# Tao model
sentiment_model = MultimodalSentimentModel(
    vision_encoder=vision_encoder,
    projector=projector,
    llm_model=model,
    tokenizer=tokenizer,
    num_aspects=NUM_ASPECTS,
    num_sentiments=NUM_SENTIMENTS,
).to(device)

total_params, trainable_params = sentiment_model.get_trainable_params()

print(f"\nTotal parameters: {total_params:,}")


print(f"Trainable parameters: {trainable_params:,}")print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
# ============================================
# TRAINING CONFIGURATION - Dual-Head Loss
# ============================================

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Hyperparameters
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 10
LAMBDA_SENTIMENT = 1.0  # Weight cho sentiment loss

# Optimizer - chi train projection + aspect_classifier + sentiment_classifier
trainable_params_list = [p for p in sentiment_model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params_list, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Scheduler
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)

# Loss functions - Dual Head
aspect_criterion = nn.BCEWithLogitsLoss()           # Aspect detection (multi-label)
sentiment_criterion = nn.CrossEntropyLoss(reduction='none')  # Per-aspect sentiment (masked)

print(f"Training Configuration (Dual-Head):")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Weight Decay: {WEIGHT_DECAY}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Total Steps: {total_steps}")
print(f"  Lambda Sentiment: {LAMBDA_SENTIMENT}")
print(f"  Aspect Loss: BCEWithLogitsLoss (multi-label)")
print(f"  Sentiment Loss: CrossEntropyLoss (masked, per-aspect)")


In [ ]:
# ============================================
# TRAINING LOOP - Dual-Head
# ============================================

def compute_dual_loss(outputs, aspect_labels, sentiment_labels,
                      aspect_criterion, sentiment_criterion, lambda_sentiment=1.0):
    """
    Tinh total loss cho dual-head model.
    
    Args:
        outputs: dict with aspect_logits [B, 6] and sentiment_logits [B, 6, 3]
        aspect_labels: [B, 6] multi-hot
        sentiment_labels: [B, 6, 3] per-aspect sentiment one-hot
        
    Returns:
        total_loss, aspect_loss, sentiment_loss
    """
    aspect_logits = outputs["aspect_logits"]      # [B, 6]
    sentiment_logits = outputs["sentiment_logits"]  # [B, 6, 3]
    
    # === ASPECT LOSS: BCE (multi-label) ===
    aspect_loss = aspect_criterion(aspect_logits.float(), aspect_labels.float())
    
    # === SENTIMENT LOSS: CrossEntropy (masked, chi tren aspects xuat hien) ===
    B, A, S = sentiment_logits.shape  # B, 6, 3
    
    # Flatten: [B*6, 3] va [B*6]
    sent_logits_flat = sentiment_logits.view(B * A, S)              # [B*6, 3]
    sent_targets_flat = sentiment_labels.view(B * A, S).argmax(-1)  # [B*6] -> class indices
    mask = aspect_labels.view(B * A)                                 # [B*6] -> 0 or 1
    
    # CrossEntropy per sample (reduction='none')
    ce_loss = sentiment_criterion(sent_logits_flat.float(), sent_targets_flat)  # [B*6]
    
    # Mask: chi tinh loss tren aspects thuc su xuat hien
    if mask.sum() > 0:
        sentiment_loss = (ce_loss * mask).sum() / mask.sum()
    else:
        sentiment_loss = torch.tensor(0.0, device=aspect_logits.device)
    
    total_loss = aspect_loss + lambda_sentiment * sentiment_loss
    
    return total_loss, aspect_loss, sentiment_loss


def train_epoch(model, dataloader, optimizer, scheduler,
                aspect_criterion, sentiment_criterion, device, lambda_sentiment=1.0):
    """Train 1 epoch (Dual-Head)."""
    model.train()
    total_loss = 0
    total_aspect_loss = 0
    total_sentiment_loss = 0
    num_batches = 0
    
    progress_bar = tqdm(dataloader, desc="Training")
    
    for batch in progress_bar:
        # Inputs
        pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        aspect_labels = batch["aspect_labels"].to(device)
        sentiment_labels = batch["sentiment_labels"].to(device)
        
        optimizer.zero_grad()
        
        # Forward: Image + Text -> Dual-Head outputs
        outputs = model(pixel_values, input_ids, attention_mask)
        
        # Dual loss
        loss, a_loss, s_loss = compute_dual_loss(
            outputs, aspect_labels, sentiment_labels,
            aspect_criterion, sentiment_criterion, lambda_sentiment
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        total_aspect_loss += a_loss.item()
        total_sentiment_loss += s_loss.item()
        num_batches += 1
        
        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "asp": f"{a_loss.item():.4f}",
            "sent": f"{s_loss.item():.4f}",
        })
    
    return {
        "total": total_loss / num_batches,
        "aspect": total_aspect_loss / num_batches,
        "sentiment": total_sentiment_loss / num_batches,
    }


@torch.no_grad()
def validate(model, dataloader, aspect_criterion, sentiment_criterion,
             device, lambda_sentiment=1.0):
    """Validate model (Dual-Head)."""
    model.eval()
    total_loss = 0
    total_aspect_loss = 0
    total_sentiment_loss = 0
    num_batches = 0
    
    all_aspect_preds = []
    all_aspect_labels = []
    all_sentiment_preds = []
    all_sentiment_labels = []
    all_aspect_masks = []
    
    for batch in tqdm(dataloader, desc="Validating"):
        pixel_values = batch["pixel_values"].to(device, dtype=torch.float16)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        aspect_labels = batch["aspect_labels"].to(device)
        sentiment_labels = batch["sentiment_labels"].to(device)
        
        outputs = model(pixel_values, input_ids, attention_mask)
        
        loss, a_loss, s_loss = compute_dual_loss(
            outputs, aspect_labels, sentiment_labels,
            aspect_criterion, sentiment_criterion, lambda_sentiment
        )
        
        total_loss += loss.item()
        total_aspect_loss += a_loss.item()
        total_sentiment_loss += s_loss.item()
        num_batches += 1
        
        # Aspect predictions (sigmoid > 0.5)
        aspect_preds = (torch.sigmoid(outputs["aspect_logits"]) > 0.5).cpu()
        all_aspect_preds.append(aspect_preds)
        all_aspect_labels.append(aspect_labels.cpu())
        
        # Sentiment predictions (argmax per aspect)
        sentiment_preds = outputs["sentiment_logits"].argmax(dim=-1).cpu()  # [B, 6]
        all_sentiment_preds.append(sentiment_preds)
        all_sentiment_labels.append(sentiment_labels.cpu())
        all_aspect_masks.append(aspect_labels.cpu())
    
    avg_losses = {
        "total": total_loss / num_batches,
        "aspect": total_aspect_loss / num_batches,
        "sentiment": total_sentiment_loss / num_batches,
    }
    
    all_aspect_preds = torch.cat(all_aspect_preds, dim=0)      # [N, 6]
    all_aspect_labels = torch.cat(all_aspect_labels, dim=0)    # [N, 6]
    all_sentiment_preds = torch.cat(all_sentiment_preds, dim=0)  # [N, 6]
    all_sentiment_labels = torch.cat(all_sentiment_labels, dim=0)  # [N, 6, 3]
    all_aspect_masks = torch.cat(all_aspect_masks, dim=0)       # [N, 6]
    
    return avg_losses, {
        "aspect_preds": all_aspect_preds,
        "aspect_labels": all_aspect_labels,
        "sentiment_preds": all_sentiment_preds,
        "sentiment_labels": all_sentiment_labels,
        "aspect_masks": all_aspect_masks,
    }


In [ ]:
# ============================================
# RUN TRAINING (Dual-Head) - Save to output3/
# ============================================

import os
os.makedirs("output3", exist_ok=True)

best_val_loss = float('inf')
train_losses = []
val_losses = []

print("Starting Dual-Head Training...")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 30)
    
    train_loss_dict = train_epoch(
        sentiment_model, train_loader, optimizer, scheduler,
        aspect_criterion, sentiment_criterion, device, LAMBDA_SENTIMENT
    )
    train_losses.append(train_loss_dict)
    
    val_loss_dict, val_results = validate(
        sentiment_model, dev_loader,
        aspect_criterion, sentiment_criterion, device, LAMBDA_SENTIMENT
    )
    val_losses.append(val_loss_dict)
    
    print(f"Train Loss: {train_loss_dict['total']:.4f} "
          f"(aspect={train_loss_dict['aspect']:.4f}, sentiment={train_loss_dict['sentiment']:.4f})")
    print(f"Val Loss:   {val_loss_dict['total']:.4f} "
          f"(aspect={val_loss_dict['aspect']:.4f}, sentiment={val_loss_dict['sentiment']:.4f})")
    
    if val_loss_dict['total'] < best_val_loss:
        best_val_loss = val_loss_dict['total']
        # Save TRAINABLE parameters (projector + aspect_classifier + sentiment_classifier)
        from safetensors.torch import save_file
        trainable_state = {
            k: v.cpu() for k, v in sentiment_model.state_dict().items()
            if any(k.startswith(prefix) for prefix in [
                'projector.', 'aspect_classifier.', 'sentiment_classifier.'
            ])
        }
        save_file(trainable_state, 'output3/best_model.safetensors')
        torch.save({
            'epoch': epoch,
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss_dict,
            'num_aspects': NUM_ASPECTS,
            'num_sentiments': NUM_SENTIMENTS,
            'aspect_labels': ASPECT_LABELS,
            'sentiment_labels': SENTIMENT_LABELS,
        }, 'output3/training_info.pt', _use_new_zipfile_serialization=True)
        print(f"[SAVED] Best model saved to output3/ ({len(trainable_state)} tensors)")

print("\n" + "=" * 60)
print("Training Complete!")
print(f"Best Validation Loss: {best_val_loss:.4f}")


In [ ]:
# ============================================
# PLOT TRAINING CURVES - Val Loss vs Train Loss
# ============================================

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

epochs_range = range(1, len(train_losses) + 1)

ax.plot(epochs_range, [l['total'] for l in train_losses], 'b-o', label='Train Loss', linewidth=2)
ax.plot(epochs_range, [l['total'] for l in val_losses], 'r-o', label='Val Loss', linewidth=2)

# Annotate values
for i, (t, v) in enumerate(zip(train_losses, val_losses)):
    ax.annotate(f"{t['total']:.4f}", (i+1, t['total']), textcoords="offset points", xytext=(0, 10), fontsize=8, ha='center', color='blue')
    ax.annotate(f"{v['total']:.4f}", (i+1, v['total']), textcoords="offset points", xytext=(0, -15), fontsize=8, ha='center', color='red')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Train Loss vs Val Loss', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# %pip install scikit-learn

---
## 7. Evaluation (Dual-Head: Aspect Detection + Per-Aspect Sentiment + Combined)

3 bo metrics:
1. **Aspect Detection**: P/R/F1 cho viec detect aspect nao xuat hien (multi-label)
2. **Per-Aspect Sentiment**: Accuracy/F1 cho sentiment prediction tren cac aspects da detect
3. **Combined Aspect#Sentiment**: So sanh predicted pairs vs ground truth pairs (18-class)


In [ ]:
# ============================================
# EVALUATION METRICS - Dual-Head
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    hamming_loss,
)
import numpy as np


def compute_aspect_metrics(aspect_preds, aspect_labels, id2aspect):
    """
    Tinh metrics cho Aspect Detection (multi-label).
    
    Args:
        aspect_preds: [N, 6] binary predictions
        aspect_labels: [N, 6] ground truth
    """
    preds_np = aspect_preds.numpy()
    labels_np = aspect_labels.numpy()
    
    metrics = {
        "micro_precision": precision_score(labels_np, preds_np, average='micro', zero_division=0),
        "micro_recall": recall_score(labels_np, preds_np, average='micro', zero_division=0),
        "micro_f1": f1_score(labels_np, preds_np, average='micro', zero_division=0),
        "macro_precision": precision_score(labels_np, preds_np, average='macro', zero_division=0),
        "macro_recall": recall_score(labels_np, preds_np, average='macro', zero_division=0),
        "macro_f1": f1_score(labels_np, preds_np, average='macro', zero_division=0),
        "hamming_loss": hamming_loss(labels_np, preds_np),
        "exact_match_ratio": np.mean(np.all(preds_np == labels_np, axis=1)),
    }
    return metrics


def compute_sentiment_accuracy(sentiment_preds, sentiment_labels, aspect_masks):
    """
    Tinh accuracy cho Sentiment prediction chi tren cac aspects thuc su xuat hien.
    
    Args:
        sentiment_preds: [N, 6] argmax indices (0=Pos, 1=Neg, 2=Neu)
        sentiment_labels: [N, 6, 3] one-hot ground truth
        aspect_masks: [N, 6] ground truth aspect mask (1 = aspect xuat hien)
    """
    # Convert one-hot sentiment labels to class indices
    sent_true = sentiment_labels.argmax(dim=-1)  # [N, 6]
    
    mask = aspect_masks.bool()  # [N, 6]
    
    if mask.sum() == 0:
        return {"accuracy": 0.0, "total_aspects": 0}
    
    # Chi so sanh tren aspects thuc su xuat hien
    correct = ((sentiment_preds == sent_true) & mask).sum().item()
    total = mask.sum().item()
    
    return {
        "accuracy": correct / total,
        "correct": int(correct),
        "total_aspects": int(total),
    }


def compute_combined_metrics(aspect_preds, sentiment_preds, aspect_labels,
                              sentiment_labels, id2aspect, id2sentiment):
    """
    Tinh metrics cho Combined Aspect#Sentiment (18-class multi-label).
    
    So sanh predicted (aspect, sentiment) pairs vs ground truth pairs.
    """
    N = aspect_preds.shape[0]
    num_aspects = aspect_preds.shape[1]
    num_sentiments = len(id2sentiment)
    num_combined = num_aspects * num_sentiments  # 18
    
    # Build combined multi-hot vectors [N, 18]
    combined_preds = torch.zeros(N, num_combined)
    combined_labels = torch.zeros(N, num_combined)
    
    sent_true = sentiment_labels.argmax(dim=-1)  # [N, 6]
    
    for i in range(N):
        for a in range(num_aspects):
            # Ground truth
            if aspect_labels[i, a] == 1:
                s_true = sent_true[i, a].item()
                combined_labels[i, a * num_sentiments + s_true] = 1.0
            
            # Predictions
            if aspect_preds[i, a] == 1:
                s_pred = sentiment_preds[i, a].item()
                combined_preds[i, a * num_sentiments + s_pred] = 1.0
    
    preds_np = combined_preds.numpy()
    labels_np = combined_labels.numpy()
    
    metrics = {
        "micro_precision": precision_score(labels_np, preds_np, average='micro', zero_division=0),
        "micro_recall": recall_score(labels_np, preds_np, average='micro', zero_division=0),
        "micro_f1": f1_score(labels_np, preds_np, average='micro', zero_division=0),
        "macro_precision": precision_score(labels_np, preds_np, average='macro', zero_division=0),
        "macro_recall": recall_score(labels_np, preds_np, average='macro', zero_division=0),
        "macro_f1": f1_score(labels_np, preds_np, average='macro', zero_division=0),
        "samples_f1": f1_score(labels_np, preds_np, average='samples', zero_division=0),
        "hamming_loss": hamming_loss(labels_np, preds_np),
        "exact_match_ratio": np.mean(np.all(preds_np == labels_np, axis=1)),
    }
    
    return metrics, combined_preds, combined_labels


def print_dual_head_metrics(aspect_metrics, sentiment_acc, combined_metrics):
    """In tat ca metrics cho Dual-Head model."""
    print("\n" + "=" * 70)
    print("DUAL-HEAD EVALUATION RESULTS")
    print("=" * 70)
    
    print("\n[1] ASPECT DETECTION (multi-label, 6 aspects)")
    print(f"  Micro  - P: {aspect_metrics['micro_precision']:.4f}  "
          f"R: {aspect_metrics['micro_recall']:.4f}  F1: {aspect_metrics['micro_f1']:.4f}")
    print(f"  Macro  - P: {aspect_metrics['macro_precision']:.4f}  "
          f"R: {aspect_metrics['macro_recall']:.4f}  F1: {aspect_metrics['macro_f1']:.4f}")
    print(f"  Hamming Loss: {aspect_metrics['hamming_loss']:.4f}")
    print(f"  Exact Match:  {aspect_metrics['exact_match_ratio']:.4f}")
    
    print(f"\n[2] PER-ASPECT SENTIMENT (on GT aspects)")
    print(f"  Accuracy: {sentiment_acc['accuracy']:.4f} "
          f"({sentiment_acc.get('correct', '?')}/{sentiment_acc['total_aspects']} aspects)")
    
    print(f"\n[3] COMBINED Aspect#Sentiment (18-class multi-label)")
    print(f"  Micro  - P: {combined_metrics['micro_precision']:.4f}  "
          f"R: {combined_metrics['micro_recall']:.4f}  F1: {combined_metrics['micro_f1']:.4f}")
    print(f"  Macro  - P: {combined_metrics['macro_precision']:.4f}  "
          f"R: {combined_metrics['macro_recall']:.4f}  F1: {combined_metrics['macro_f1']:.4f}")
    print(f"  Samples F1:   {combined_metrics['samples_f1']:.4f}")
    print(f"  Hamming Loss: {combined_metrics['hamming_loss']:.4f}")
    print(f"  Exact Match:  {combined_metrics['exact_match_ratio']:.4f}")
    
    print("=" * 70)


In [ ]:
# ============================================
# EVALUATE ON TEST SET (Dual-Head)
# ============================================

from safetensors.torch import load_file
import os

# Load best model from output3/
model_path = 'output3/best_model.safetensors'
info_path = 'output3/training_info.pt'

if os.path.exists(model_path):
    trainable_state = load_file(model_path)
    trainable_state = {k: v.to(device) for k, v in trainable_state.items()}
    sentiment_model.load_state_dict(trainable_state, strict=False)
    if os.path.exists(info_path):
        info = torch.load(info_path, weights_only=True)
        print(f"Loaded best model from epoch {info['epoch'] + 1}")
        print(f"  Val loss: {info['val_loss']}")
    else:
        print("Loaded best model from output3/")
else:
    print("[WARNING] No saved model found in output3/!")

# Evaluate on test set
test_losses, test_results = validate(
    sentiment_model, test_loader,
    aspect_criterion, sentiment_criterion, device, LAMBDA_SENTIMENT
)

print(f"\nTest Loss: {test_losses['total']:.4f} "
      f"(aspect={test_losses['aspect']:.4f}, sentiment={test_losses['sentiment']:.4f})")

# Compute all 3 sets of metrics
aspect_metrics = compute_aspect_metrics(
    test_results["aspect_preds"], test_results["aspect_labels"], ID2ASPECT
)

sentiment_acc = compute_sentiment_accuracy(
    test_results["sentiment_preds"], test_results["sentiment_labels"],
    test_results["aspect_masks"]
)

combined_metrics, combined_preds, combined_labels = compute_combined_metrics(
    test_results["aspect_preds"], test_results["sentiment_preds"],
    test_results["aspect_labels"], test_results["sentiment_labels"],
    ID2ASPECT, ID2SENTIMENT
)

print_dual_head_metrics(aspect_metrics, sentiment_acc, combined_metrics)


In [ ]:
# ============================================
# PER-CLASS METRICS (Dual-Head)
# ============================================

def compute_per_aspect_metrics(aspect_preds, aspect_labels, id2aspect):
    """Per-aspect detection metrics."""
    preds_np = aspect_preds.numpy()
    labels_np = aspect_labels.numpy()
    
    print("\n" + "=" * 80)
    print("PER-ASPECT DETECTION METRICS")
    print("=" * 80)
    print(f"{'Aspect':<20} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-" * 80)
    
    for idx in range(preds_np.shape[1]):
        name = id2aspect[idx]
        y_true = labels_np[:, idx]
        y_pred = preds_np[:, idx]
        support = int(y_true.sum())
        if support == 0:
            p = r = f = 0.0
        else:
            p = precision_score(y_true, y_pred, zero_division=0)
            r = recall_score(y_true, y_pred, zero_division=0)
            f = f1_score(y_true, y_pred, zero_division=0)
        print(f"{name:<20} {p:>10.4f} {r:>10.4f} {f:>10.4f} {support:>10}")
    print("=" * 80)


def compute_per_combined_metrics(combined_preds, combined_labels, id2combined):
    """Per Aspect#Sentiment combined metrics (18 classes)."""
    preds_np = combined_preds.numpy()
    labels_np = combined_labels.numpy()
    
    print("\n" + "=" * 80)
    print("PER ASPECT#SENTIMENT METRICS (18 classes)")
    print("=" * 80)
    print(f"{'Label':<30} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-" * 80)
    
    for idx in range(preds_np.shape[1]):
        name = id2combined[idx]
        y_true = labels_np[:, idx]
        y_pred = preds_np[:, idx]
        support = int(y_true.sum())
        if support == 0:
            p = r = f = 0.0
        else:
            p = precision_score(y_true, y_pred, zero_division=0)
            r = recall_score(y_true, y_pred, zero_division=0)
            f = f1_score(y_true, y_pred, zero_division=0)
        print(f"{name:<30} {p:>10.4f} {r:>10.4f} {f:>10.4f} {support:>10}")
    print("=" * 80)


# Run per-class metrics
compute_per_aspect_metrics(test_results["aspect_preds"], test_results["aspect_labels"], ID2ASPECT)
compute_per_combined_metrics(combined_preds, combined_labels, ID2COMBINED)


---
## 8. Inference voi Trained Dual-Head Pipeline

Su dung trained pipeline: Image -> Swin V2 -> Projection -> LLM -> Dual Head -> Aspect + Sentiment

Output: Detected aspects + sentiment cho moi aspect
VD: `{"Room": "Positive", "Service": "Negative"}`


In [ ]:
# ============================================
# INFERENCE FUNCTION - Dual-Head
# ============================================

def predict_aspect_sentiment(
    image_path: str,
    comment: str,
    model: MultimodalSentimentModel = sentiment_model,
    aspect_threshold: float = 0.5,
) -> dict:
    """
    Phan tich Aspect-Category Sentiment su dung trained Dual-Head pipeline.
    
    Pipeline: 
      Image -> Swin V2 -> Projection ──┐
                                       ├──> LLM -> Pooling ──┬──> Aspect Head -> aspects
      Text  -> Tokenizer -> Embeddings ┘                     └──> Sentiment Head -> sentiments
    
    Args:
        image_path: Duong dan toi anh
        comment: Text binh luan
        model: MultimodalSentimentModel da train (dual-head)
        aspect_threshold: Nguong sigmoid cho aspect detection (default 0.5)
    
    Returns:
        Dict chua detected aspects, per-aspect sentiments, va detailed probabilities
    """
    model.eval()
    
    # IMAGE PATH
    transform = build_transform(input_size=256)
    image = Image.open(image_path).convert("RGB")
    pixel_values = transform(image).unsqueeze(0).to(device, dtype=torch.float16)
    
    # TEXT PATH
    text_inputs = tokenizer(
        [comment], padding=True, truncation=True, max_length=256, return_tensors="pt"
    )
    input_ids = text_inputs.input_ids.to(device)
    attention_mask = text_inputs.attention_mask.to(device)
    
    # Forward: Dual-Head
    with torch.no_grad():
        outputs = model(pixel_values, input_ids, attention_mask)
        aspect_probs = torch.sigmoid(outputs["aspect_logits"]).squeeze(0).cpu()   # [6]
        sentiment_probs = torch.softmax(outputs["sentiment_logits"], dim=-1).squeeze(0).cpu()  # [6, 3]
    
    # Detect aspects
    detected_aspects = []
    aspect_sentiments = {}
    detailed = {}
    
    for a_idx in range(NUM_ASPECTS):
        aspect_name = ID2ASPECT[a_idx]
        a_prob = aspect_probs[a_idx].item()
        is_detected = a_prob > aspect_threshold
        
        # Per-aspect sentiment
        s_probs = sentiment_probs[a_idx]  # [3]
        best_s_idx = s_probs.argmax().item()
        best_sentiment = ID2SENTIMENT[best_s_idx]
        
        detailed[aspect_name] = {
            "aspect_probability": a_prob,
            "detected": is_detected,
            "sentiment_probabilities": {
                ID2SENTIMENT[s]: s_probs[s].item() for s in range(NUM_SENTIMENTS)
            },
            "predicted_sentiment": best_sentiment,
        }
        
        if is_detected:
            detected_aspects.append(aspect_name)
            aspect_sentiments[aspect_name] = best_sentiment
    
    # Fallback: neu khong co aspect nao > threshold, lay aspect co prob cao nhat
    if not detected_aspects:
        best_a_idx = aspect_probs.argmax().item()
        best_aspect = ID2ASPECT[best_a_idx]
        detected_aspects.append(best_aspect)
        detailed[best_aspect]["detected"] = True
        s_probs = sentiment_probs[best_a_idx]
        best_s_idx = s_probs.argmax().item()
        aspect_sentiments[best_aspect] = ID2SENTIMENT[best_s_idx]
    
    return {
        "detected_aspects": detected_aspects,
        "aspect_sentiments": aspect_sentiments,
        "detailed": detailed,
    }


def predict_aspect_sentiment_batch(
    image_paths: list,
    comments: list,
    model: MultimodalSentimentModel = sentiment_model,
    aspect_threshold: float = 0.5,
) -> list:
    """
    Predict Aspect-Category Sentiment cho nhieu samples cung luc.
    """
    model.eval()
    transform = build_transform(input_size=256)
    
    # Load images
    pixel_list = []
    for img_path in image_paths:
        image = Image.open(img_path).convert("RGB")
        pixel_list.append(transform(image))
    pixel_values = torch.stack(pixel_list).to(device, dtype=torch.float16)
    
    # Tokenize
    text_inputs = tokenizer(
        comments, padding=True, truncation=True, max_length=256, return_tensors="pt"
    )
    input_ids = text_inputs.input_ids.to(device)
    attention_mask = text_inputs.attention_mask.to(device)
    
    # Forward
    with torch.no_grad():
        outputs = model(pixel_values, input_ids, attention_mask)
        aspect_probs = torch.sigmoid(outputs["aspect_logits"]).cpu()           # [B, 6]
        sentiment_probs = torch.softmax(outputs["sentiment_logits"], dim=-1).cpu()  # [B, 6, 3]
    
    results = []
    for i in range(len(image_paths)):
        detected_aspects = []
        aspect_sentiments = {}
        
        for a_idx in range(NUM_ASPECTS):
            if aspect_probs[i, a_idx].item() > aspect_threshold:
                aspect_name = ID2ASPECT[a_idx]
                detected_aspects.append(aspect_name)
                best_s = sentiment_probs[i, a_idx].argmax().item()
                aspect_sentiments[aspect_name] = ID2SENTIMENT[best_s]
        
        if not detected_aspects:
            best_a = aspect_probs[i].argmax().item()
            aspect_name = ID2ASPECT[best_a]
            detected_aspects.append(aspect_name)
            best_s = sentiment_probs[i, best_a].argmax().item()
            aspect_sentiments[aspect_name] = ID2SENTIMENT[best_s]
        
        results.append({
            "detected_aspects": detected_aspects,
            "aspect_sentiments": aspect_sentiments,
        })
    
    return results


In [ ]:
# ============================================
# DEMO INFERENCE - Dual-Head Pipeline
# ============================================

import matplotlib.pyplot as plt

# Lay 1 sample tu test set
demo_sample = test_dataset.samples[0]
demo_image_path = demo_sample["image_path"]
demo_comment = demo_sample["comment"]
demo_true_labels = demo_sample["raw_labels"]

print(f"Comment: {demo_comment}")
print(f"True labels: {demo_true_labels}")
print()

# Hien thi anh
img = Image.open(demo_image_path)
plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.axis('off')
plt.title(f"True: {demo_true_labels}")
plt.tight_layout()
plt.show()

# Predict su dung trained Dual-Head pipeline
result = predict_aspect_sentiment(demo_image_path, demo_comment)

print("=" * 60)
print("PREDICTION (Dual-Head Pipeline):")
print("=" * 60)
print(f"\n  Detected Aspects: {result['detected_aspects']}")
print(f"\n  Aspect-Sentiments:")
for aspect, sentiment in result['aspect_sentiments'].items():
    print(f"    {aspect}: {sentiment}")

print(f"\n  Detailed Probabilities:")
for aspect, info in result['detailed'].items():
    marker = ">>>" if info["detected"] else "   "
    sent_str = ", ".join([f"{s}: {p:.3f}" for s, p in info['sentiment_probabilities'].items()])
    print(f"  {marker} {aspect:<15} (p={info['aspect_probability']:.3f}) -> [{sent_str}] -> {info['predicted_sentiment']}")
print("=" * 60)


---
## Tong Ket

### Kien truc Dual-Head:

```
                                    (( Aspect Logits [B, 6] ))    (( Sentiment Logits [B, 6, 3] ))
                                              ^                              ^
                                              |                              |
                                    [ Aspect Classifier ]        [ Sentiment Classifier ]
                                       (trainable)                    (trainable)
                                              ^                              ^
                                              |                              |
                                              +──────────┬───────────────────+
                                                         |
                                                  ( Mean Pooling )
                                                         ^
                                                  ( Hidden states )
                                                         |
    +----------------------------------------------------+
    |                    LLM (frozen)                     |
    +----------------------------------------------------+
         ^                               ^
         |                               |
   (Lang tokens)                   (Lang tokens)
   [B, 64, 4096]                   [B, N, 4096]
         |                               |
    [ Projection ]  (trainable)   [  Embeddings  ]  (frozen)
         ^                               ^
    (Img tokens)                   (word-pieces)
   [B, 64, 1024]                         |
         |                               |
    [  Swin V2   ]  (frozen)      [  Tokenizer   ]
         ^                               ^
   (4x4 patches)                         |
         |                               |
    [Preprocessor]                       |
         ^                               |
         |                               |
  ((   Image   ))                 (( Input text  ))
```

### Components:

| Component | Input | Output | Trainable |
|-----------|-------|--------|-----------|
| Preprocessor | Image (any size) | Tensor [3, 256, 256] | - |
| Swin Transformer V2 | Tensor [3, 256, 256] | Img tokens [64, 1024] | Frozen |
| Projection (MLP) | Img tokens [64, 1024] | Lang tokens [64, 4096] | **Trainable** |
| Tokenizer | Text | Word-pieces | - |
| Embeddings | Word-pieces | Lang tokens [N, 4096] | Frozen |
| LLM (InternLM3-8B) | Lang tokens | Hidden states | Frozen |
| **Aspect Classifier** | Pooled hidden [4096] | Aspect logits [6] | **Trainable** |
| **Sentiment Classifier** | Pooled hidden [4096] | Sentiment logits [6, 3] | **Trainable** |

### Dataset: ViMACSA

| Split | Samples |
|-------|---------|
| Train | ~3000 |
| Dev | ~500 |
| Test | ~500 |

### Labels:

| Type | Values |
|------|--------|
| **Aspects** (6) | Facilities, Public_area, Location, Food, Room, Service |
| **Sentiments** (3) | Positive, Negative, Neutral |
| **Combined** (18) | Aspect#Sentiment (VD: Room#Positive, Service#Negative) |

### Loss Functions:

| Head | Loss | Giai thich |
|------|------|------------|
| Aspect Head | `BCEWithLogitsLoss` | Multi-label: nhieu aspects co the xuat hien cung luc |
| Sentiment Head | `CrossEntropyLoss` (masked) | Per-aspect: moi aspect chi co 1 sentiment, chi tinh tren aspects xuat hien |
| Total | `aspect_loss + λ * sentiment_loss` | λ = 1.0 (co the tune) |

### Output (output3/):

- `best_model.safetensors` - Trainable weights (projector + aspect_classifier + sentiment_classifier)
- `training_info.pt` - Training metadata
